# Exploration of WVV ÖPNV data using the GTFS-RT database

No real goal, just exploring the data while printing and plotting as much relevant information as possible.

In [12]:
import requests
import pandas as pd
from google.transit import gtfs_realtime_pb2

# Configuration
URL = "https://realtime.gtfs.de/realtime-free.pb"

In [13]:
def get_gtfs_rt_dataframe(url):
    # Fetch the realtime feed
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed to fetch data: {response.status_code}")
        return pd.DataFrame()

    # Parse the protobuf data
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    # Extract relevant data into a list of dictionaries
    data = []
    for entity in feed.entity:
        # Check for 'trip_update' because that's where trip_ids live
        if entity.HasField('trip_update'):
            trip = entity.trip_update.trip
            
            # Basic trip info
            row = {
                "trip_id": trip.trip_id,
                "route_id": trip.route_id,
                "start_time": trip.start_time,
                "start_date": trip.start_date,
                "schedule_relationship": trip.schedule_relationship,
                "timestamp": entity.trip_update.timestamp
            }
            
            # Add the delay from the first stop update if available
            if entity.trip_update.stop_time_update:
                update = entity.trip_update.stop_time_update[0]
                # Check for arrival or departure delay
                row["delay"] = update.arrival.delay if update.HasField('arrival') else update.departure.delay
            
            data.append(row)

    # Convert to pandas DataFrame
    return pd.DataFrame(data)

In [17]:
# Execution
df = get_gtfs_rt_dataframe(URL)

# Check for specific trip_ids
with open('wvv_trip_ids.txt', 'r') as f:
    target_ids = [line.strip() for line in f]
filtered_df = df[df['trip_id'].isin(target_ids)]

print(f"Total trips found: {len(df)}")
print("\nFiltered Results:")
print(filtered_df.loc[filtered_df["delay"] > 0].head())

Total trips found: 37907

Filtered Results:
       trip_id route_id start_time start_date  schedule_relationship  \
2194    823102                       20260406                      0   
4874    567235                       20260406                      0   
13903    37122                       20260406                      0   
14828  1039637                       20260406                      0   
15083  1000024                       20260406                      0   

       timestamp   delay  
2194           0   203.0  
4874           0   194.0  
13903          0   187.0  
14828          0  1606.0  
15083          0   725.0  
